In [1]:
import sys, site
print("Python executable:", sys.executable)
print(sys.version)
print("Site packages:", site.getsitepackages())

Python executable: c:\Users\Apar2\OneDrive\Desktop\College\Spring 2026\635\project\voice-mood-music-recommender\moody-env\Scripts\python.exe
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Site packages: ['c:\\Users\\Apar2\\OneDrive\\Desktop\\College\\Spring 2026\\635\\project\\voice-mood-music-recommender\\moody-env', 'c:\\Users\\Apar2\\OneDrive\\Desktop\\College\\Spring 2026\\635\\project\\voice-mood-music-recommender\\moody-env\\Lib\\site-packages']


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
import warnings
import sklearn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

warnings.filterwarnings("ignore")

print(torch.__version__)
print(torch.__file__)
print("Import successful!")


2.11.0+cpu
c:\Users\Apar2\OneDrive\Desktop\College\Spring 2026\635\project\voice-mood-music-recommender\moody-env\Lib\site-packages\torch\__init__.py
Import successful!


In [3]:
# Load features
X = np.load('../data/X_features.npy')
y = np.load('../data/y_labels.npy')

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

print(f'Classes: {le.classes_}')
print(f'Encoded: {np.unique(y_enc)}')

Classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
Encoded: [0 1 2 3 4 5 6 7]


In [4]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')
print(f'Feature size: {X_train.shape[1]}')

Train size: 2304
Test size:  576
Feature size: 112


In [5]:
class CNN(nn.Module):
    def __init__(self, input_size=112, num_classes=8):
        super(CNN, self).__init__()
        # For tabular data, use fully connected layers
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)
        return x

In [6]:
# Check number of classes first
print(f'Number of unique labels: {len(np.unique(y_enc))}')
print(f'Label range: {y_enc.min()} to {y_enc.max()}')

Number of unique labels: 8
Label range: 0 to 7


In [7]:
# Convert to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).long()
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).long()

# Get actual dimensions and number of classes
input_size = X_train_tensor.shape[1]
num_classes = len(np.unique(y_train))


# Initialize model with correct dimensions
cnn = CNN(input_size=input_size, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(cnn.parameters(), lr=0.001, momentum=0.9)

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=32, shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=32, shuffle=False
)

for epoch in range(20):
    epoch_loss = 0.0
    num_batches = 0
    
    for i, (inputs, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = cnn(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1

        if i % 10 == 9:
            loss = 0.0

In [8]:
# Evaluating the Model

correct = 0
total = 0

# Calculate predictions and accuracy and f1 score
with torch.no_grad():
    for data in test_loader:
        inputs, labels = data
        outputs = cnn(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    print(f'Accuracy: {accuracy:.4f}')

# Get all predictions and true labels for classification report
all_preds = []
all_labels = []
with torch.no_grad():
    for data in test_loader:
        inputs, labels = data
        outputs = cnn(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:", classification_report(all_labels, all_preds, target_names=le.classes_))
print("F1 Score:", f1_score(all_labels, all_preds, average='weighted'))

    

Accuracy: 0.3663
Classification Report:               precision    recall  f1-score   support

       angry       0.43      0.57      0.49        76
        calm       0.36      0.84      0.50        77
     disgust       0.18      0.14      0.16        77
     fearful       0.40      0.38      0.39        77
       happy       0.24      0.13      0.17        77
     neutral       0.00      0.00      0.00        38
         sad       0.18      0.13      0.15        77
   surprised       0.36      0.30      0.33        77

    accuracy                           0.33       576
   macro avg       0.27      0.31      0.27       576
weighted avg       0.29      0.33      0.29       576

F1 Score: 0.29099434370403976
